<a href="https://colab.research.google.com/github/edgardlt03/ICO-Trabajos/blob/main/Vector_Stores_y_B%C3%BAsqueda_Sem%C3%A1ntica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers scikit-learn kagglehub --quiet


In [2]:
from sentence_transformers import SentenceTransformer

### Carga del Animal Fun Facts Dataset

Fuente: https://github.com/ekohrt/animal-fun-facts-dataset


In [25]:
import csv, io, urllib.request
RAW_URL = (
    "https://raw.githubusercontent.com/ekohrt/"
    "animal-fun-facts-dataset/main/animal-fun-facts-dataset.csv"
)

with urllib.request.urlopen(RAW_URL) as resp:
    csv_text = resp.read().decode("utf-8")

animal_docs: list[Document] = []
for row in csv.DictReader(io.StringIO(csv_text)):
    text = row.get("text", "").strip()
    if text:
        animal_docs.append(Document(
            text=text,
            metadata={
                "animal_name":    row.get("animal_name", "").strip(),
                "source":         row.get("source", "").strip(),
                "media_link":     row.get("media_link", "").strip(),
                "wikipedia_link": row.get("wikipedia_link", "").strip(),
            }
        ))

print(f"Documentos cargados: {len(animal_docs):,}")
print("Metadata:", animal_docs[50].metadata)


Documentos cargados: 7,731
Metadata: {'animal_name': 'bat', 'source': 'https://www.animalfactsencyclopedia.com/Bat-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Bat'}


## Parte I: VectorStore básico

In [48]:
import numpy as np

class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata

class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document

class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        texts = [doc.text for doc in documents]
        new_embs = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True)
        self.embeddings = new_embs if self.embeddings is None else np.vstack([self.embeddings, new_embs])

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if not self.documents:
            return []
        q_emb = self.model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
        scores = cosine_similarity(q_emb, self.embeddings)[0]
        top_indices = np.argsort(scores)[::-1][:min(top_k, len(self.documents))]
        return [SearchResult(float(scores[i]), self.documents[i]) for i in top_indices]


### Instancia de VectorStore y carga de documentos

In [27]:
model = SentenceTransformer("all-MiniLM-L6-v2")  # 384 dimensiones

vs = VectorStore(model)
vs.add_documents(animal_docs)

print(f"Documentos : {len(vs.documents):,}")
print(f"Embeddings : {vs.embeddings.shape}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/242 [00:00<?, ?it/s]

Documentos : 7,731
Embeddings : (7731, 384)


### 5 consultas de ejemplo score, texto y metadatos


In [54]:
def show(results: list[SearchResult]):

    for i, r in enumerate(results, 1):
        print(f"{i}.Score       : {r.score:.4f}")
        print(f"  Texto       : {r.document.text[:120]}")
        print(f"  animal_name : {r.document.metadata['animal_name']}")
        print(f"  source      : {r.document.metadata['source']}")
        print(f"  wikipedia   : {r.document.metadata['wikipedia_link']}")
        print(f"  media_link  : {r.document.metadata['media_link']}\n")



In [63]:
print("CONSULTA 1\n")
show(vs.search("animals that sleep for many hours a day", top_k=3))
print("\nCONSULTA 2\n")
show(vs.search("dolphin communication and intelligence", top_k=3))
print("\nCONSULTA 3\n")
show(vs.search("poisonous animals that can kill humans", top_k=3))
print("\nCONSULTA 4\n")
show(vs.search("birds that migrate thousands of miles", top_k=3))
print("\nCONSULTA 5\n")
show(vs.search("fastest animal on land", top_k=3))

CONSULTA 1

1.Score       : 0.7291
  Texto       : They often sleep 16 hours a day!.
In addition to being solitary animals, armadillos also like to sleep—a lot.
  animal_name : armadillo
  source      : https://factanimal.com/armadillo/
  wikipedia   : /wiki/Armadillo
  media_link  : 

2.Score       : 0.7173
  Texto       : These animals are diurnal, sleeping in treetop leaves and branches during the night. They spend most of day in search of
  animal_name : coatimundi
  source      : https://seaworld.org/animals/facts/mammals/coatimundi/
  wikipedia   : /wiki/Coati
  media_link  : 

3.Score       : 0.7156
  Texto       : Anteaters sleep as much as 15 hours each day.
  animal_name : giant anteater
  source      : https://seaworld.org/animals/facts/mammals/giant-anteater/
  wikipedia   : /wiki/Giant_anteater
  media_link  : 


CONSULTA 2

1.Score       : 0.6805
  Texto       : Dolphins communicate with clicks and whistles..
This helps them navigate, warn of potential predators and hunt 

##Parte II: Filtering by metadata

In [ ]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass